# FORGE — Deepfake Detection (EfficientNet-B4)

Train an image-level deepfake classifier for the FORGE `/detect-deepfake` endpoint.

**Environment**: Kaggle P100, PyTorch, mixed-precision
**Backbone**: `tf_efficientnet_b4_ns` from `timm` (~19M params)
**Target**: ≥ 85% validation accuracy

**Inputs**: add `deepfake-detection-challenge` (sample) as a Kaggle input dataset, OR `faceforensicspp` if available.

**Outputs**: `model.pth` + `config.json` saved to `/kaggle/working/`. Download both and commit to `checkpoints/deepfake/` in the FORGE repo.

## 1. Setup

In [ ]:
!pip -q install timm==0.9.16 facenet-pytorch==2.5.3 albumentations==1.4.4 pillow-heif==0.16.0

In [ ]:
import os, json, time, random, glob
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import timm
from facenet_pytorch import MTCNN
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu')

## 2. Data prep — extract face crops from DFDC videos

We sample N frames per video and run MTCNN to extract the dominant face.  
Saves crops to `/kaggle/working/faces/{real,fake}/` and produces a balanced index.

In [ ]:
# Adjust this if you mounted a different DFDC subset.
DFDC_ROOT = Path('/kaggle/input/deepfake-detection-challenge')
TRAIN_VIDEO_DIRS = sorted(DFDC_ROOT.glob('train_sample_videos*')) or [DFDC_ROOT / 'train_sample_videos']
META_PATHS = []
for d in TRAIN_VIDEO_DIRS:
    m = d / 'metadata.json'
    if m.exists():
        META_PATHS.append((d, m))
print('found video dirs:', [str(d) for d, _ in META_PATHS])

In [ ]:
FACES_DIR = Path('/kaggle/working/faces'); FACES_DIR.mkdir(exist_ok=True)
(FACES_DIR / 'real').mkdir(exist_ok=True); (FACES_DIR / 'fake').mkdir(exist_ok=True)

FRAMES_PER_VIDEO = 4   # sample 4 evenly-spaced frames per clip
MAX_VIDEOS_PER_LABEL = 1500  # cap to fit memory/time on P100
FACE_SIZE = 224

mtcnn = MTCNN(image_size=FACE_SIZE, margin=20, post_process=False, device=DEVICE, keep_all=False)

def sample_frames(video_path, n=FRAMES_PER_VIDEO):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return []
    idxs = np.linspace(0, total - 1, n).astype(int)
    frames = []
    for i in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(i))
        ok, frame = cap.read()
        if not ok: continue
        frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    cap.release()
    return frames

def crop_face(rgb_frame):
    pil = Image.fromarray(rgb_frame)
    face = mtcnn(pil)
    if face is None:
        return None
    arr = face.cpu().numpy().transpose(1, 2, 0).clip(0, 255).astype('uint8')
    return arr

def label_videos(meta_path, video_dir):
    with open(meta_path) as f: meta = json.load(f)
    out = []
    for fn, info in meta.items():
        label = 1 if info.get('label') == 'FAKE' else 0
        out.append((str(video_dir / fn), label))
    return out

all_videos = []
for d, m in META_PATHS:
    all_videos.extend(label_videos(m, d))
print('total videos referenced:', len(all_videos))

# Cap per class for balanced sampling.
by_class = {0: [], 1: []}
random.shuffle(all_videos)
for v, y in all_videos:
    if len(by_class[y]) < MAX_VIDEOS_PER_LABEL:
        by_class[y].append((v, y))
videos = by_class[0] + by_class[1]
random.shuffle(videos)
print('using:', Counter(y for _, y in videos))

rows = []
t0 = time.time()
for i, (vp, y) in enumerate(videos):
    if not Path(vp).exists(): continue
    frames = sample_frames(vp)
    for j, fr in enumerate(frames):
        face = crop_face(fr)
        if face is None: continue
        sub = 'fake' if y == 1 else 'real'
        out_path = FACES_DIR / sub / f"{Path(vp).stem}_{j}.jpg"
        cv2.imwrite(str(out_path), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))
        rows.append({'path': str(out_path), 'label': y})
    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{len(videos)} videos · {len(rows)} faces · {time.time()-t0:.0f}s')

df = pd.DataFrame(rows)
print('total faces extracted:', len(df))
print(df['label'].value_counts())
df.to_csv('/kaggle/working/faces_index.csv', index=False)

## 3. Splits + augmentation

In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv('/kaggle/working/faces_index.csv')
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=SEED)
print('train:', len(train_df), 'val:', len(val_df))

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.4),
    A.ImageCompression(quality_lower=50, quality_upper=100, p=0.5),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.3),
    A.Normalize(MEAN, STD),
    ToTensorV2(),
])
val_tf = A.Compose([A.Resize(224, 224), A.Normalize(MEAN, STD), ToTensorV2()])

class FaceDataset(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        img_data = cv2.imread(row['path'])
        if img_data is None:
            return torch.zeros(3, 224, 224), torch.tensor(row['label'], dtype=torch.float32)
        img = cv2.cvtColor(img_data, cv2.COLOR_BGR2RGB)
        x = self.tf(image=img)['image']
        return x, torch.tensor(row['label'], dtype=torch.float32)

train_ds = FaceDataset(train_df, train_tf)
val_ds   = FaceDataset(val_df,   val_tf)
BATCH = 24
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## 4. Model + training loop

In [ ]:
model = timm.create_model('tf_efficientnet_b4_ns', pretrained=True, num_classes=1).to(DEVICE)
criterion = nn.BCEWithLogitsLoss()
optim_ = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
EPOCHS = 8
sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim_, T_max=EPOCHS)
scaler = GradScaler()

def run_epoch(dl, train=True):
    model.train(train)
    losses, ys, ps = [], [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x).squeeze(1)
            loss = criterion(logits, y)
        if train:
            optim_.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optim_); scaler.update()
        losses.append(loss.item())
        ys.append(y.detach().cpu().numpy())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
    y_all = np.concatenate(ys); p_all = np.concatenate(ps)
    pred = (p_all > 0.5).astype(int)
    acc = accuracy_score(y_all, pred)
    pre, rec, f1, _ = precision_recall_fscore_support(y_all, pred, average='binary', zero_division=0)
    auc = roc_auc_score(y_all, p_all) if len(set(y_all)) == 2 else float('nan')
    return float(np.mean(losses)), acc, pre, rec, f1, auc, y_all, p_all

best = {'val_acc': 0.0, 'epoch': -1}
for epoch in range(EPOCHS):
    t0 = time.time()
    tr_loss, tr_acc, *_ = run_epoch(train_dl, train=True)
    val_loss, val_acc, val_pre, val_rec, val_f1, val_auc, y_v, p_v = run_epoch(val_dl, train=False)
    sched.step()
    print(f"[{epoch+1}/{EPOCHS}]  train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
          f"val loss {val_loss:.4f} acc {val_acc:.3f} P {val_pre:.3f} R {val_rec:.3f} F1 {val_f1:.3f} AUC {val_auc:.3f} | {time.time()-t0:.0f}s")
    if val_acc > best['val_acc']:
        best.update(val_acc=val_acc, epoch=epoch, val_f1=val_f1, val_auc=val_auc)
        torch.save(model.state_dict(), '/kaggle/working/model.pth')
        print(f"   ↳ saved best (val_acc={val_acc:.3f})")
print('best:', best)

## 5. Final eval + threshold tuning

In [ ]:
# Reload best weights and run final validation.
model.load_state_dict(torch.load('/kaggle/working/model.pth', map_location=DEVICE))
_, val_acc, val_pre, val_rec, val_f1, val_auc, y_v, p_v = run_epoch(val_dl, train=False)
print('FINAL VAL — acc', val_acc, 'F1', val_f1, 'AUC', val_auc)
print('confusion:\n', confusion_matrix(y_v, (p_v > 0.5).astype(int)))

# Pick threshold that maximizes F1 on validation set (small grid).
best_thr, best_f1 = 0.5, 0.0
for t in np.linspace(0.3, 0.7, 41):
    pred = (p_v > t).astype(int)
    _, _, f1, _ = precision_recall_fscore_support(y_v, pred, average='binary', zero_division=0)
    if f1 > best_f1:
        best_f1, best_thr = f1, float(t)
print(f'best threshold: {best_thr:.3f}  (F1={best_f1:.3f})')

## 6. Export config.json

In [ ]:
config = {
  'model_name': 'tf_efficientnet_b4_ns',
  'input_size': 224,
  'mean': MEAN,
  'std': STD,
  'threshold': best_thr,
  'val_accuracy': float(val_acc),
  'val_f1': float(val_f1),
  'val_auc': float(val_auc),
  'classes': ['REAL', 'DEEPFAKE'],
  'note': 'state_dict only; load with timm.create_model("tf_efficientnet_b4_ns", num_classes=1)'
}
with open('/kaggle/working/config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(json.dumps(config, indent=2))
print('\nFiles ready in /kaggle/working/:')
for p in ['model.pth', 'config.json']:
    fp = Path('/kaggle/working') / p
    if fp.exists(): print(f'  {p}  ({fp.stat().st_size/1024/1024:.1f} MB)')

## 7. Deploy

1. Download `model.pth` and `config.json` from the Kaggle output.
2. Place them in `checkpoints/deepfake/` of the FORGE repo.
3. Restart the FastAPI server — it will pre-warm the detector on startup.
4. Sanity check: `curl -F file=@some_face.jpg http://localhost:7860/detect-deepfake`